# Variational Inference in Context

This chapter is adapted from AM207 (Harvard, Fall 2021). [Watch the recorded lecture](https://youtu.be/uzG8gKJOp_w).

::::{admonition} In this chapter
:class: tip

1. How to Evaluate Approximate Inference
2. How to Improve Approximate Inference
3. Why do I care?
::::

In [13]:
### Import basic libraries
import numpy
import autograd.numpy as np
import autograd.numpy.random as npr
import autograd.scipy.stats.multivariate_normal as mvn
import autograd.scipy.stats.norm as norm
from autograd import grad
from autograd.misc.optimizers import adam
import numpy
import scipy as sp
import pandas as pd
import sklearn as sk
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import rc
from IPython.display import HTML
from IPython.display import YouTubeVideo
%matplotlib inline

## How to Evaluate Approximate Inference

### How Good is Your Variational Approximation of the True Posterior?
**Question:** Why is it hard to check that the variational posterior is a good approximation of the true posterior?

You can't simply visualize the true posterior, since it's 1) high dimensional, 2) intractable to sample form.

**Question:** What if we computed the KL-divergence between the true posterior and the varaitional approximation?

["Practical Posterior Error Bounds from Variational Objectives"](https://arxiv.org/abs/1910.04102): Unfortunately, even when the KL-divergence between $q$ and $p$ is effectively zero, the difference between the means and variances of $q$ and $p$ can be **arbitrarily large**. That is, a small KL-divergence doesn't capture our intuition about what it means for two distributions to be similar.

### Alternative Posterior Evaluation Metrics
In ["Yes, But Did It Work: Evaluating Variational Inference"](https://arxiv.org/abs/1802.02538), the authors proposes two alternative posterior evaluation metrics:

**1. The Pareto Smoothed Importance Sampling Diagnostic**: If the variational approximation $q$ is very different than $p$, then the importance sampling MC estimate of $\mathbb{E}_{p(\theta|\text{Data})}\left[ f(\theta) \right]$,

$$
\mathbb{E}_{p(\theta|\text{Data})}\left[ f(\theta) \right] \approx \frac{1}{S}\sum_{s=1}^S \frac{p(\theta|\text{Data})}{q(\theta)} f(\theta_s),\; \theta_s \sim q(\theta)
$$

will have very high variance, due to the fact that the importance weights $\frac{p(\theta|\text{Data})}{q(\theta)}$ will be very heterogeneous. Thus, methods of smoothing the weights, like ***Pareto Smoothed Importance Sampling (PSIS)*** will have poor performance. The efficacy of PSIS can be used as a diagnostic tool for testing if $q$ is similar to $p$.

**2. The Variational Simulation-Based Calibration (VSBC) Diagnostic**: Given a Bayesian model $p(y, \theta) = p(y|\theta) p(\theta)$, if your variational inference proceedure is good at approximating the posterior $p(\theta|y)$ then it should do well for most sets of data that is generated from the model $p(y, \theta)$.

We generate $M$ sets of synthetic data from $p(y, \theta)$. For each set of the data we approximate the posterior $q_m(\theta)$. We then perform a calibration test on each variational approximation. This will reveal if your variational inference procedure is biased - produces approximation that are consistently flawed in some specific way.

## How to Improve Approximate Inference

### Your Variational Approximation Sucks, Can You Fix It?
The current research on improving variational inference can be categorize by which **design choice** each work tries to improve:

**1. Choice of Divergence:** In {doc}`Lab 5 </labs/lab-05-kl-divergence-pymc3>`, you explored the draw-backs of fitting an approximate posterior $q$ to the true posterior $p$ using KL-divergence. There are a huge number of works that explore performing variational inference using other types of divergences, for example: <br>
$\quad$ **a.** ["Black-box $\alpha$-divergence Minimization"](https://arxiv.org/abs/1511.03243)<br>
$\quad$ **b.** ["Stein Variational Gradient Descent"](https://arxiv.org/abs/1608.04471)

**2. Choice of Variational Family:** In the {ref}`Chapter 1 problem set <ps-hw0>`, you've seen that even for a simple Bayesian linear regression model, the model parameters were corrlated in the posterior. In {doc}`Lab 8 </labs/lab-08-vi-vs-hmc>`, you've seen that minimizing the KL-divergence to fit an isotropic Gaussian means that you can only capture one mode in the posterior. There are a huge number of works that explore using different types of variational families:<br>
$\quad$ **a.** ["Variational Inference with Normalizing Flows"](https://arxiv.org/abs/1505.05770)<br>
$\quad$ **b.** ["The Variational Gaussian Process"](https://arxiv.org/abs/1511.06499)<br>

**3. Choice of Optimization Procedure:** Even if your divergence measure and variational family are well-chosen, the optimization objective can still be non-convex! This means that your optimization procedure might return a $q$ that is a only local-optimum. There are a large number of works that address how to jump out of local optima using SGD and some works that specifically address the optimization challenges of variational inference:<br>
$\quad$ **a.** ["Proximity Variational Inference"](https://arxiv.org/abs/1705.08931)

### Are There Any Good Properties of Variational Approximations that We Are Sure About?
In ["Frequentist Consistency of Variational Bayes"](https://arxiv.org/abs/1705.03439), the authors prove that under **certain assumptions** on $p(\theta)$ and $p(y|\theta)$:
1.  as the number of observation increases, the variational approximation converges (in Total Variation distance) to the $q$ that minimizes the KL-divergence to a normal distribution centered at the ground truth parameters $\theta_{\text{true}}$ that generated the data.<br><br>
2. the mean of the variational approximation is consistent and asymptoptically normal.

Unfortunately, the assumptions required by the theorems **do not hold for neural network likelihood models**.

### Why is Inference for Neural Networks Hard?
Research on the likelihoods (and hence posteriors) of neural networks are beginning to give us ways of visualizing these high-dimensional functions using low-dimensional (3D) projections. [Loss Landsacapes](https://losslandscape.com) represents some of the latest efforts at visualization:


```{figure} ../figures/loss.jpg
:name: fig-19-loss
:width: 100%
:align: center
```

### It's Weirder Than You Can Imagine
In ["Loss Landscape Sightseeing with Multi-Point Optimization"](https://arxiv.org/pdf/1910.03867.pdf), the authors show that neural network likelihoods (and thus posteriors) are so complicated that you can find a 2-D projection that can create any pattern you want:


```{figure} ../figures/loss2.jpg
:name: fig-19-loss2
:width: 100%
:align: center
```


**The lesson**: take every low-dimensional visualization with a grain of salt (i.e. it far from accurately represents the entire landscape).

## But why do I care?

### Evaluation of Variational Approximation for Real Down-stream Tasks
In practice, you may only care about the quality of your posterior approximation in so far as it affects model performance on your down-stream task. So what do we want from our machine learning or statistical models, especially when they are used in safety, or fairness critical applications (e.g. personalized medicine, health-care resource allocation, criminal justic systems)?

Most of us in the community agree that we want models that *1)* makes accurate predictions *2)* gives realistic estimates of its prective uncertainty. So that the model can be held-accountable by humans in the system. There are a number of subcomunities in ML that focus on studying the social impact of machine learning models as well as how to design models whose negative impact can be mitigated.

**a.** ["Concrete Problems in AI Safety"](https://arxiv.org/pdf/1606.06565.pdf)<br>
**b.** ["Predict Responsibly: Improving Fairness and Accuracy by Learning to Defer"](https://papers.nips.cc/paper/7853-predict-responsibly-improving-fairness-and-accuracy-by-learning-to-defer.pdf)

### Predictiveness of the Variational Approximate Posterior
We can check the predictiveness of our varational approximation of the posterior by sampling models from the posterior and then making predictions using these models by sampling from the likelihood. We then check that these predictions align well with observed data by:

1. **Visualization:** visualizing the posterior predictive against the observed data. *This is generally impossible for high dimensional data.*<br><br>

2. **Log Marginal Data Likelihood:** we compute the expected value of the log likelihood of the test data under our approximate posterior. *Log likelihood is only useful for comparing two different models, given a single model it is hard to say if a log likelihood value is "good enough".*<br><br>

3. **Expected Mean Square Error and Accuracy:** we compute the expected MSE (for regression) or accuracy (for classification) on test data under our approximate posterior. *While these metric frames model quality in concrete task related terms, they are each misleading when the data contains outliers or imbalanced classes.*

### Quality of Variational Approximate Posterior Predictive Uncertainty
We've argued in this course that good predictive uncertainty must involve an accurate assessment of both ***epistemic*** and ***aleatoric*** uncertainty. Since each type of uncertainty requires a different risk-management action in the down-stream task.


```{figure} ../figures/posterior_pred.jpg
:name: fig-19-posterior-pred
:width: 100%
:align: center
```


Unfortunately, there isn't a single good statistical metric for assessing the quality of the uncertainty of a model. "Good" epstemic uncertainty is especially hard to quantify. Rather than looking for statistical tests of uncertainty, some in the ML community advocate for assessing model uncertainty with respect to a set of benchmark down-stream tasks: [Bayesian Deep Learning Benchmarks](https://github.com/OATML/bdl-benchmarks)

### Does a Poor Posterior Approximation Imply a Poor Posterior Predictive?
You've seen in the {ref}`Chapter 18 problem set <ps-hw7>` that even with an HMC sampler that is far from converged, you were able to produce posterior predictives that aligned well with the data and had good epistemic and aleatoric uncertainty. In fact, a number of works are showing that by capturing a little piece (if it is the right piece) of the true posterior of a BNN, you are able to capture most of the variation you want in the posterior predictive.

In ["Subspace Inference for Bayesian Deep Learning"](https://arxiv.org/pdf/1907.07504.pdf), the authors provide toy examples where one obtains a posterior predictive distribution with good epistemic and aleatoric uncertainty by reducing a high dimensional parameter space to 2-dimensions and performing inference in the 2-dimensional subspace:

```{figure} ../figures/posterior_pred2.jpg
:name: fig-19-posterior-pred2
:width: 100%
:align: center
```

## New Developments in Deep Bayes

### What's Wrong with Posterior Approximations for BNNs
***(joint work with Jiayu Yao, Soumya Ghosh, Finale Doshi-Velez)***

**The motivation:** Frequently, we introduce a new approximate inference method for BNNs based on our intuition of what properties of the true posterior are important to include in our approximate posterior. We then measure the ***log-likelihood*** of our learned model on test data.

**The idea:** We evaluate the ability of a number of state-of-the-art approximate inference methods for BNN on synthetic data that can be visualized to see if various custom approximations of the posterior translates to desirable posterior predictives.

We also test if the commonly used evaluation metrics, like test log-likelihood, can distinguish high quality posterior predictves from poor quality ones.

***From:*** [Quality of Uncertainty Quantification for Bayesian Neural Network Inference](https://arxiv.org/pdf/1906.09686.pdf)

### What Did We Learn?
**Take-away:** Metric like test log-likelihood measures how well the approximate posterior predictive aligns with the data, not how well it approximate the true posterior predictive.


```{figure} ../figures/cubic_compare.png
:name: fig-19-cubic-compare
:width: 100%
:align: center
```


**Take-away:** Training with data gaps and evaluating test likelihood can only catch problems if the true function is complex in these gaps.


```{figure} ../figures/complex_compare.png
:name: fig-19-complex-compare
:width: 100%
:align: center
```


**Take-away:** Inference methods that specialize in approximating specific properties in the posterior do not necessarily produce desirable posterior predictives.

### What's Wrong with the Predictions from Mean-Field Variational Posteriors of BNNs?
***(joint work with Beau Coker, Finale Doshi-Velez)***

The problem with mean-field VI for BNNs isn't just with the variance. In [Wide Mean-Field Variational Bayesian Neural Networks Ignore the Data](https://arxiv.org/pdf/2106.07052.pdf). we show that as the network width becomes wider, the posterior predictive mean ignores the data completely and learns a constant (flat) function!


```{figure} ../figures/widebnn.png
:name: fig-19-widebnn
:width: 518px
:align: center
```


The fact that the posterior predictive mean of mean-field variational posteriors of wide BNNs ***underfit*** the data has been observed empirically in [Overpruning in Variational Bayesian Neural Networks](https://arxiv.org/abs/1801.06230).

### Relationships Between Deep Ensembles and Bayesian Neural Networks
Althhough we've been thinking about ensembles and Bayesian models as belonging to a dichotomy (frequentist vs Bayesian statistics), in papers like [Conservative Uncertainty Estimation By Fitting Prior Networks](https://openreview.net/pdf?id=BJlahxHYDS) and [Repulsive Deep Ensembles are Bayesian](https://arxiv.org/abs/2106.11642), we see that ensembles and Bayesian models have a deep relationship.

**Take-away:** In fact, some ensembles can be interpreted as collections of samples from a corresponding Bayesian model!

### Alternative Models for Uncertainty Quantification
Although mean-field VI is touted as a fast way of performing inference on BNNs, they are still hard to scale to truly large networks. For this reason, alternative deep Bayesian models have been gaining popularity. **Neural Linear Models** is a model that learns deterministic weights for an NN except for the last layer, where we place priors and perform exact Bayesian inference:


```{figure} ../figures/Unknown-20.png
:name: fig-19-unknown-20
:width: 100%
:align: center
```


***From:*** [Learned Uncertainty-Aware (LUNA) Bases for Bayesian Regression using Multi-Headed Auxiliary Networks](https://arxiv.org/pdf/2006.11695.pdf)

### Predictive Uncertainties of Alternative Deep Bayesian Models May Not be Better
***(joint work with Sujay Thakur, Cooper Lorsung, Yaniv Yacoby, Finale Doshi-Velez)***

In [Learned Uncertainty-Aware (LUNA) Bases for Bayesian Regression using Multi-Headed Auxiliary Networks](https://arxiv.org/pdf/2006.11695.pdf), we show that the posterior predictive uncertainty of NLMs are not what we might want. The problem has a lot to do with how we typically train neural networks.


```{figure} ../figures/Unknown-18.png
:name: fig-19-unknown-18
:width: 100%
:align: center
```

### What Uncertainties Do We Need in Deep Learning?
More and more folks in the Deep Bayes community is coming to the conclusion that the quality of uncertainty estimation can only be assessed in reference to a specific down-stream task, e.g. active learning, continual learning, Bayesian optimization, out of distribution detection, learning to defer (rejection learning), model interpretation etc.


```{figure} ../figures/bayesopt.png
:name: fig-19-bayesopt
:width: 360px
:align: center
```


**Take-away:** There is no universal definition of or metric for "good" uncertainty - "good" uncertainty is uncertainty that is useful for the task.

**Take-away:** Uncertainty estimation that is good for one task might be bad for another!

## Summary

What did we observe in the {ref}`Chapter 22 problem set <ps-hw8>`? We setup a BNN with 1 hidden layer and 5 hidden nodes.
1. Did your HMC converge? Do you feel confident that you can get it to converge?
    - Did the posterior samples give you desirable predictive uncertainty?<br><br>
    
2. Did your BBVI optimize the ELBO fully? How do you know that you've optimized your ELBO fully?
    - Did the variational posterior give you desirable predictive uncertainty?<br><br>
    
3. When the network is large, do you believe that either HMC or mean-field Gaussian VI will give you a good approximation of the posterior?
    - Do we necessarily need a good approximation of the posterior to get desirable uncertainty?<br><br>
    
4. **The Training Wheels are Off!** When doing machine learning in the wild, no one knows the "right answer", your metrics are not absolute, the only thing you can rely on is your intuition (which is you internalizing the theory you've learned as well as your hands-on experience working with these models).
    - How can we know that our HMC has no bugs in it?
    - How can we know that our BBVI has no bugs in it?
    - How can we know that we've fully optimized anything (e.g. the ELBO) through gradient descent?
    - What are the impediments to BBVI optimizing fully? I.e. what do we tune to improve performance?
    - What are the impediments to HMC mixing? I.e. what do we tune to improve performance? <br><br>
    
5. **What this class is not:** This class is not a one-stop for all the tools you need to be a fluent practitioner of in-demand state-of-the art machine learning. It's normal that:
    - You still don't feel like you're comfortable impplementing all the models from scratch and deploying it in real-life conditions. 
      You should **not** be comfortable now, but you will be with practice in the future (you're getting some of that practice with the project)!
    - You don't remember everything from the class, and that as time passes, you seem to forget more.
      You'll forget many if not most facts, but when you re-learn these facts, understanding will come more easily and with more depth.<br><br>

6. **What is this class:** This class is a map or atlas and paints a broad landscape of modern-day probabilistic machine learning. Using the structure of this class you can better organize and process new ML knowledge. 

  This class is a foundation for you to build on, future classes in ML **will** reference these models, techniques and concepts:
    - Reinforcement learning (Harvard)
    - Causal inference (Harvard)
    - ML for Healthcare (MIT)
    - Neural Processing (Harvard)
    - Special Topics in ML (Harvard)
    - Interpretable ML (Harvard)
    - AI for Social Good (Harvard)

## New Developments in Deep Bayes



### What's Wrong with Posterior Approximations for BNNs
***(joint work with Jiayu Yao, Soumya Ghosh, Finale Doshi-Velez)***

**The motivation:** Frequently, we introduce a new approximate inference method for BNNs based on our intuition of what properties of the true posterior are important to include in our approximate posterior. We then measure the ***log-likelihood*** of our learned model on test data.

**The idea:** We evaluate the ability of a number of state-of-the-art approximate inference methods for BNN on synthetic data that can be visualized to see if various custom approximations of the posterior translates to desirable posterior predictives.

We also test if the commonly used evaluation metrics, like test log-likelihood, can distinguish high quality posterior predictves from poor quality ones.

***From:*** [Quality of Uncertainty Quantification for Bayesian Neural Network Inference](https://arxiv.org/pdf/1906.09686.pdf)

### What Did We Learn?
**Take-away:** Metric like test log-likelihood measures how well the approximate posterior predictive aligns with the data, not how well it approximate the true posterior predictive.


```{figure} ../figures/cubic_compare.png
:name: fig-19-summary-cubic-compare
:width: 100%
:align: center
```


**Take-away:** Training with data gaps and evaluating test likelihood can only catch problems if the true function is complex in these gaps.


```{figure} ../figures/complex_compare.png
:name: fig-19-summary-complex-compare
:width: 100%
:align: center
```


**Take-away:** Inference methods that specialize in approximating specific properties in the posterior do not necessarily produce desirable posterior predictives.

### What's Wrong with the Predictions from Mean-Field Variational Posteriors of BNNs?
***(joint work with Beau Coker, Finale Doshi-Velez)***

The problem with mean-field VI for BNNs isn't just with the variance. In [Wide Mean-Field Variational Bayesian Neural Networks Ignore the Data](https://arxiv.org/pdf/2106.07052.pdf). we show that as the network width becomes wider, the posterior predictive mean ignores the data completely and learns a constant (flat) function!


```{figure} ../figures/widebnn.png
:name: fig-19-summary-widebnn
:width: 518px
:align: center
```


The fact that the posterior predictive mean of mean-field variational posteriors of wide BNNs ***underfit*** the data has been observed empirically in [Overpruning in Variational Bayesian Neural Networks](https://arxiv.org/abs/1801.06230).

### Relationships Between Deep Ensembles and Bayesian Neural Networks
Althhough we've been thinking about ensembles and Bayesian models as belonging to a dichotomy (frequentist vs Bayesian statistics), in papers like [Conservative Uncertainty Estimation By Fitting Prior Networks](https://openreview.net/pdf?id=BJlahxHYDS) and [Repulsive Deep Ensembles are Bayesian](https://arxiv.org/abs/2106.11642), we see that ensembles and Bayesian models have a deep relationship.

**Take-away:** In fact, some ensembles can be interpreted as collections of samples from a corresponding Bayesian model!

### Alternative Models for Uncertainty Quantification
Although mean-field VI is touted as a fast way of performing inference on BNNs, they are still hard to scale to truly large networks. For this reason, alternative deep Bayesian models have been gaining popularity. **Neural Linear Models** is a model that learns deterministic weights for an NN except for the last layer, where we place priors and perform exact Bayesian inference:


```{figure} ../figures/Unknown-20.png
:name: fig-19-summary-unknown-20
:width: 100%
:align: center
```


***From:*** [Learned Uncertainty-Aware (LUNA) Bases for Bayesian Regression using Multi-Headed Auxiliary Networks](https://arxiv.org/pdf/2006.11695.pdf)

### Predictive Uncertainties of Alternative Deep Bayesian Models May Not be Better
***(joint work with Sujay Thakur, Cooper Lorsung, Yaniv Yacoby, Finale Doshi-Velez)***

In [Learned Uncertainty-Aware (LUNA) Bases for Bayesian Regression using Multi-Headed Auxiliary Networks](https://arxiv.org/pdf/2006.11695.pdf), we show that the posterior predictive uncertainty of NLMs are not what we might want. The problem has a lot to do with how we typically train neural networks.


```{figure} ../figures/Unknown-18.png
:name: fig-19-summary-unknown-18
:width: 100%
:align: center
```

### What Uncertainties Do We Need in Deep Learning?
More and more folks in the Deep Bayes community is coming to the conclusion that the quality of uncertainty estimation can only be assessed in reference to a specific down-stream task, e.g. active learning, continual learning, Bayesian optimization, out of distribution detection, learning to defer (rejection learning), model interpretation etc.


```{figure} ../figures/bayesopt.png
:name: fig-19-summary-bayesopt
:width: 360px
:align: center
```


**Take-away:** There is no universal definition of or metric for "good" uncertainty - "good" uncertainty is uncertainty that is useful for the task.

**Take-away:** Uncertainty estimation that is good for one task might be bad for another!

## Further Reading

**Approximate Inference for Deep Bayesian Models**

- (Using an Alternate Divergence Measure for VI) [Black-Box α-Divergence Minimization](https://arxiv.org/pdf/1511.03243.pdf)
- (Using an Alternate Variational Family) [Variational Inference with Normalizing Flows](https://arxiv.org/pdf/1505.05770.pdf)
- (Approximating the Posterior in a Subspace) [Subspace Inference for Bayesian Deep Learning](http://auai.org/uai2019/proceedings/papers/435.pdf)

**Evaluating Approximate Inference for Deep Bayesian Models**

- (Why are BNN Posteriors So Difficult?) [Visualizing the Loss Landscape of Neural Nets](https://arxiv.org/pdf/1712.09913.pdf)
- (How to Evaluate the Epistemic Uncertainty of Approximate Posteriors of BNNs) [Quality of Uncertainty Quantification for Bayesian Neural Network Inference](https://arxiv.org/pdf/1906.09686.pdf)
- (Hard Limitations of Mean-Field VI for BNNs) [On the Expressiveness of Approximate Inference in Bayesian Neural Networks](https://arxiv.org/pdf/1909.00719.pdf)

**Alternate Models for Deep Bayesian Networks**

- (Neural Linear Models) [Learned Uncertainty-Aware (LUNA) Bases for Bayesian](https://arxiv.org/pdf/2006.11695.pdf)
- (Manifold Gaussian Processes) [Manifold Gaussian Processes for Regression](https://arxiv.org/pdf/1402.5876.pdf)